# 11 하이브리드 수요예측 — SBC vs ML scheme 비교

**2-type(고변동 E · 저변동 C)** 에 대해, 10장과 동일한 **LSTM + Best 임베딩** 예측으로
**SBC(rule-base) vs ML(TS2Vec+KMeans) 클러스터링 scheme**을 **제품수 가중 WMAPE**로 비교합니다.

## 검증할 가설

학위논문은 **수요 변동성에 따라 유리한 scheme이 갈린다**고 보고했습니다.

| 변동성 | 학위논문 결과 | 근거 |
|---|---|---|
| 저변동 (Center A) | **ML 우세** | 수요가 안정적이라 임베딩 군집의 세분화가 효과적 |
| 고변동 (Center B) | **SBC 우세** | ADI·CV² 규칙 분류가 불규칙 수요를 더 잘 구분 |

본 실습은 같은 가설을 Ecuador 데이터의 저변동 **C**(System CV 0.259)·고변동 **E**(0.429)에 적용해
**동일한 대응이 나타나는지** 확인합니다. 지표는 논문과 같은 제품수 가중 WMAPE = Σ n_k·MAPE_k / Σ n_k 입니다.

→ 결과는 아래 ② 해석 참조. **본 실습에서는 이 대응이 재현되지 않습니다.**

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 792 | 조건: 12


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import validation_weights, build_family_from_phase2_best, thesis_wmape_by_type
from utils.stats_summary import type_variation_table
from utils.config import selected_type_list

# 조건별 XGBoost + Best 임베딩(10장) 제품 단위 결과
val_weights = validation_weights(df)
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

# 논문식 WMAPE = Σ n_k·MAPE_k / Σ n_k (클러스터 제품수 가중, §4.5 Eq.50)
type_wmape = thesis_wmape_by_type(family_final)
print('=== type별 SBC vs ML (제품수 가중 WMAPE) ===')
display(type_wmape)
print('scheme 우세:', type_wmape['better_scheme'].value_counts().to_dict())

# System-Level CV(고/저변동 판별 지표)와 대조
var = type_variation_table(df)[['sys_cv', 'sku_mean_cv']].round(3)
sel = selected_type_list()  # [고변동, 저변동]
out = type_wmape.join(var)
out['variation'] = np.where(out.index == sel[0], 'high(고변동)',
                    np.where(out.index == sel[1], 'low(저변동)', ''))
print('=== scheme 우세 × System-Level CV ===')
display(out[['variation', 'sys_cv', 'SBC_wmape', 'ML_wmape', 'better_scheme']])

=== type별 SBC vs ML (제품수 가중 WMAPE) ===


,SBC_wmape,ML_wmape,delta_SBC_minus_ML,better_scheme
type,,,,
C,38.18,38.19,-0.01,SBC
E,46.72,47.60,-0.88,SBC


scheme 우세: {'SBC': 2}
=== scheme 우세 × System-Level CV ===


,variation,sys_cv,SBC_wmape,ML_wmape,better_scheme
type,,,,,
C,low(저변동),0.259,38.18,38.19,SBC
E,high(고변동),0.429,46.72,47.60,SBC


### ② 해석 — SBC vs ML scheme (제품수 가중 WMAPE)

**WMAPE = Σ_k n_k·MAPE_k / Σ_k n_k** (클러스터 제품수 가중, 논문 §4.5 Eq.50) · base = **LSTM + Best 임베딩**

#### 결과 — 양 type 모두 SBC 우세
| type | System CV | SBC WMAPE | ML WMAPE | 우세 | 논문 가설 |
|------|-----------|-----------|----------|------|-----------|
| **C** (저변동) | 0.259 | **38.18** | 38.19 | SBC (−0.01) | 저변동→ML |
| **E** (고변동) | 0.429 | **46.72** | 47.60 | **SBC** (−0.88) | 고변동→SBC ✅ |

→ 고변동 E는 논문 가설대로 SBC가 우세하지만, **저변동 C에서도 SBC가 앞서** 논문의 변동성↔scheme 대응(저변동→ML)은 재현되지 않았습니다. 다만 C의 격차는 **0.01로 사실상 동률**입니다.

#### 왜 ML scheme이 힘을 못 썼는가
ML 클러스터링 결과가 **C 30:3 · E 31:2** 로 심하게 불균형합니다. 66 시계열 규모에서는 K=2가 상한이고, 그마저도 "메가셀러 소수 vs 롱테일 다수"로 갈려 **세분화 이점이 거의 없습니다**. 제품수 가중 WMAPE는 큰 군집이 지배하므로, 사실상 롱테일 61개를 한 덩어리로 예측하는 것과 다르지 않습니다.

반면 SBC는 ADI·CV² 규칙으로 4분류하므로 계열 수가 적어도 유형별 구분이 유지됩니다. **계열이 충분해야 ML 클러스터링이 의미 있는 군집을 만든다**는 점을 역으로 보여주는 결과입니다.

#### 시사점
- **소규모 데이터에서는 규칙 기반 분류(SBC)가 안정적**입니다. ML 임베딩 군집은 계열 폭이 넓을 때 이점이 나옵니다.
- scheme 선택 시 변동성(System-Level CV)뿐 아니라 **군집 균형(min_cluster_size)** 을 함께 확인해야 합니다.
- 학위논문(12,661 SKU) 규모에서는 저변동→ML·고변동→SBC 대응이 성립했습니다. 결론이 갈리는 것은 **표본 규모 차이의 귀결**입니다.

#### 한계 (반드시 명시)
> C의 격차는 **0.01**로 무의미한 수준이고, E도 0.88에 불과합니다(66 시계열·13주 test). 이 결과로 scheme 우열을 일반화할 수 없으며, **데이터 규모와 군집 균형에 따라 결론이 달라진다**는 점이 핵심입니다.